In [2]:
import psycopg2
import pandas as pd

In [3]:
# Connect to PostgreSQL and fetch data
conn = psycopg2.connect("dbname=sui_indexer user=postgres password=56904628")
cur = conn.cursor()

In [28]:
cur.execute("""
SELECT
    (SELECT COUNT(*) FROM transactions) AS total_tx,
    (SELECT SUM(user_tx_count) FROM checkpoints) AS user_tx,
    (SELECT SUM(system_tx_count) FROM checkpoints) AS system_tx;
""")

total_tx, user_tx, system_tx = cur.fetchone()

print("Total transactions:", total_tx)
print("Total user tx:", user_tx)
print("Total system tx:", system_tx)


Total transactions: 779434
Total user tx: 8
Total system tx: 779426


In [18]:
cur.execute("""SELECT
        t.transaction_digest,
        oc.address,
        c.timestamp
    FROM transactions t
    JOIN checkpoints c
        ON t.checkpoint_sequence = c.sequence_number
    LEFT JOIN object_changes oc
        ON t.transaction_digest = oc.transaction_digest
    WHERE
        oc.change_type IN ('mutated', 'created', 'deleted')
        OR
        t.inputs_shared_mut > 0
        OR t.inputs_receiving > 0
        OR t.inputs_imm_or_owned > 0
    ORDER BY c.timestamp DESC;""")
rows = cur.fetchall()

write_transactions = []
write_object_addresses = []
write_timestamps = []

for tx_digest, address, timestamp in rows:
    write_transactions.append(tx_digest)
    write_object_addresses.append(address)
    write_timestamps.append(timestamp)


In [20]:
len(write_transactions)

6

In [21]:
cur.execute("""SELECT
    t.transaction_digest,
    c.timestamp,
    oc.address
FROM transactions t
JOIN checkpoints c
    ON t.checkpoint_sequence = c.sequence_number
LEFT JOIN object_changes oc
    ON t.transaction_digest = oc.transaction_digest
WHERE
    -- First: transaction does NOT write anything
    t.inputs_shared_mut = 0
    AND t.inputs_receiving = 0
    AND t.inputs_funds_withdrawal = 0
    AND (
        oc.change_type IS NULL             -- no object modifications
        OR oc.change_type NOT IN ('mutated', 'created', 'deleted')
    )
    AND
    (
        -- These make it a READ:
        t.inputs_pure > 0
        OR t.inputs_shared_ro > 0
        OR t.inputs_imm_or_owned > 0
    )
ORDER BY c.timestamp DESC;""")
rows = cur.fetchall()

read_transactions = []
read_object_addresses = []
read_timestamps = []

for tx_digest, address, timestamp in rows:
    read_transactions.append(tx_digest)
    read_object_addresses.append(address)
    read_timestamps.append(timestamp)


In [23]:
len(read_transactions)

3